# Notebook 02: Statistical Hypothesis Testing, Target Leakage Proofs & Customer Segmentation (Module 1)
**Project 02 — Enterprise Predictive Intelligence & Applied Machine Learning System**


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.vq import kmeans2

DATA_RAW = Path("data/raw") if Path("data/raw").exists() else Path("../data/raw")
df_c = pd.read_csv(DATA_RAW / "customer_retention_features.csv")
df_d = pd.read_csv(DATA_RAW / "delivery_delay_training.csv")
df_dem = pd.read_csv(DATA_RAW / "daily_product_demand.csv")

g1 = df_c[df_c["no_purchase_next_90d"] == 1]
g0 = df_c[df_c["no_purchase_next_90d"] == 0]
stat_rows = []
for col in ["orders_last_180d", "tenure_months", "days_since_last_order", "support_tickets_last_180d", "discount_order_share", "returns_last_180d", "sessions_last_30d", "avg_order_value"]:
    u_stat, u_p = stats.mannwhitneyu(g1[col], g0[col])
    t_stat, t_p = stats.ttest_ind(g1[col], g0[col], equal_var=False)
    r_p = df_c[col].corr(df_c["no_purchase_next_90d"], method="pearson")
    r_s = df_c[col].corr(df_c["no_purchase_next_90d"], method="spearman")
    stat_rows.append({
        "feature": col,
        "retained_mean": round(g0[col].mean(), 3),
        "lapsed_mean": round(g1[col].mean(), 3),
        "pearson_r": round(r_p, 4),
        "spearman_rho": round(r_s, 4),
        "welch_t": round(float(t_stat), 2),
        "mannwhitney_p": f"{u_p:.2e}"
    })
print("=== CUSTOMER RETENTION HYPOTHESIS TESTS (N = 30,000) ===")
print(pd.DataFrame(stat_rows).to_string(index=False))


=== CUSTOMER RETENTION HYPOTHESIS TESTS (N = 30,000) ===
                  feature  retained_mean  lapsed_mean  pearson_r  spearman_rho  welch_t mannwhitney_p
         orders_last_180d          4.404        2.811    -0.2601       -0.2660   -49.45      0.00e+00
            tenure_months         33.823       24.479    -0.2568       -0.2568   -46.44      0.00e+00
    days_since_last_order         80.543      107.634     0.2470        0.2470    44.50      0.00e+00
support_tickets_last_180d          1.128        1.326     0.0859        0.0818    14.58      1.57e-45
     discount_order_share          0.317        0.342     0.0624        0.0623    10.84      3.57e-27
        returns_last_180d          0.328        0.386     0.0470        0.0427     7.93      1.45e-13
        sessions_last_30d          5.074        4.902    -0.0364       -0.0361    -6.33      3.99e-10
          avg_order_value         82.102       83.171     0.0085        0.0045     1.46      4.38e-01


## 2. Categorical Contingency Tests ($\chi^2$), Kruskal-Wallis Seasonality & Algebraic Leakage Proofs


In [2]:
ct_w = pd.crosstab(df_d["weather_at_dispatch"], df_d["late_delivery"])
chi2_w, p_w, dof_w, _ = stats.chi2_contingency(ct_w)
ct_r = pd.crosstab(df_d["region"], df_d["late_delivery"])
chi2_r, p_r, dof_r, _ = stats.chi2_contingency(ct_r)
ct_opt = pd.crosstab(df_c["marketing_email_opt_in"], df_c["no_purchase_next_90d"])
chi2_o, p_o, _, _ = stats.chi2_contingency(ct_opt)

kw_dow, p_dow = stats.kruskal(*[grp["units_sold"].values for _, grp in df_dem.groupby("day_of_week")])
u_promo, p_promo = stats.mannwhitneyu(df_dem[df_dem["promotion_flag"]==1]["units_sold"], df_dem[df_dem["promotion_flag"]==0]["units_sold"])

print(f"Weather vs Late Delivery: Chi2 = {chi2_w:.2f} (df={dof_w}), p = {p_w:.2e}")
print(f"Region vs Late Delivery:  Chi2 = {chi2_r:.2f} (df={dof_r}), p = {p_r:.4f} (Non-significant noise)")
print(f"Email Opt-in vs Lapse:    Chi2 = {chi2_o:.4f}, p = {p_o:.4f} (Non-significant noise)")
print(f"Demand Day-of-Week Kruskal-Wallis H = {kw_dow:.2f}, p = {p_dow:.2e}")
print(f"Demand Promotion Lift Mann-Whitney U = {u_promo:.2e}, p = {p_promo:.2e}")

calc_delay = (df_d["actual_delivery_days"] - df_d["promised_delivery_days"]).clip(lower=0).round(4)
exact_delay = int((np.abs(calc_delay - df_d["delay_days"]) < 1e-4).sum())
exact_late = int(((df_d["delay_days"] > 1.0) == (df_d["late_delivery"] == 1)).sum())
print("\n=== DETERMINISTIC TARGET LEAKAGE PROOF (DELIVERY DELAY) ===")
print(f"Rows where delay_days == max(0, actual - promised): {exact_delay} / {len(df_d)} (100.00%)")
print(f"Rows where late_delivery == I(delay_days > 1.0):    {exact_late} / {len(df_d)} (100.00%)")


Weather vs Late Delivery: Chi2 = 3197.90 (df=3), p = 0.00e+00
Region vs Late Delivery:  Chi2 = 2.84 (df=4), p = 0.5857 (Non-significant noise)
Email Opt-in vs Lapse:    Chi2 = 0.0271, p = 0.8692 (Non-significant noise)
Demand Day-of-Week Kruskal-Wallis H = 4061.89, p = 0.00e+00
Demand Promotion Lift Mann-Whitney U = 3.19e+08, p = 0.00e+00

=== DETERMINISTIC TARGET LEAKAGE PROOF (DELIVERY DELAY) ===
Rows where delay_days == max(0, actual - promised): 40000 / 40000 (100.00%)
Rows where late_delivery == I(delay_days > 1.0):    40000 / 40000 (100.00%)


## 3. Customer Behavioural Segmentation (K-Means++ Clustering)


In [3]:
seg_cols = ["tenure_months", "orders_last_180d", "avg_order_value", "days_since_last_order", "discount_order_share", "returns_last_180d", "support_tickets_last_180d", "sessions_last_30d"]
X_seg = df_c[seg_cols].copy()
X_seg["avg_order_value"] = np.log1p(X_seg["avg_order_value"])
X_std = (X_seg.values - X_seg.values.mean(axis=0)) / X_seg.values.std(axis=0)
np.random.seed(42)
centroids, labels = kmeans2(X_std, 4, iter=30, minit="++")
df_c["segment"] = labels
prof = df_c.groupby("segment")[seg_cols + ["no_purchase_next_90d"]].mean().round(3)
prof["count"] = df_c.groupby("segment")["customer_id"].count()
print("=== K-MEANS++ (k=4) BEHAVIOURAL SEGMENT PROFILES ===")
print(prof.to_string())


=== K-MEANS++ (k=4) BEHAVIOURAL SEGMENT PROFILES ===
         tenure_months  orders_last_180d  avg_order_value  days_since_last_order  discount_order_share  returns_last_180d  support_tickets_last_180d  sessions_last_30d  no_purchase_next_90d  count
segment                                                                                                                                                                                             
0               15.766             1.604           82.905                 90.490                 0.326              0.261                      0.741              5.073                 0.441  11562
1               30.875             3.893           82.395                 90.962                 0.327              2.132                      1.197              5.073                 0.436   1414
2               29.482             3.416           81.917                 90.555                 0.332              0.255                      2.748           